In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StringType
from pyspark.sql.functions import col

#load data from bronze schema 
df_bronze_match_1 = spark.read.table("rugby_data_dev.rugby_bronze.match_results_raw_2015_2018")
df_bronze_match_2 = spark.read.table("rugby_data_dev.rugby_bronze.match_results_raw_2018_2026")
df_bronze_match_data = spark.read.table("rugby_data_dev.rugby_bronze.match_results_raw_2022_2026")

#drop any 'N/A', 'n/a' or blank values across all columns
df_bronze_match_1 = (df_bronze_match_1
    .dropna(subset=["HomeTeam", "AwayTeam", "Round", "HomeScore", "AwayScore"])
    .filter(~col("HomeTeam").isin(["N/A", "n/a", ""]))
    .filter(~col("AwayTeam").isin(["N/A", "n/a", ""]))
    .filter(~col("Round").isin(["N/A", "n/a", ""]))
    .filter(~col("HomeScore").isin(["N/A", "n/a", ""]))
    .filter(~col("AwayScore").isin(["N/A", "n/a", ""]))
    )

df_bronze_match_2 = (df_bronze_match_2
    .dropna(subset=["HomeTeam", "AwayTeam", "Round", "HomeScore", "AwayScore"])
    .filter(~col("HomeTeam").isin(["N/A", "n/a", ""]))
    .filter(~col("AwayTeam").isin(["N/A", "n/a", ""]))
    .filter(~col("Round").isin(["N/A", "n/a", ""]))
    .filter(~col("HomeScore").isin(["N/A", "n/a", ""]))
    .filter(~col("AwayScore").isin(["N/A", "n/a", ""]))
    )

#cast data in the 'Rounds' columns to String --> BigInt in the 2015-2018 file in bronze schema and enforce across all datasets
df_bronze_match_1 = df_bronze_match_1.withColumn("Round", col("Round").cast(StringType()))
df_bronze_match_2 = df_bronze_match_2.withColumn("Round", col("Round").cast(StringType()))
# df_bronze_match_data = df_bronze_match_data.withColumn("Round", col("Round").cast(StringType()))

#merge data in match 1 + 2 
df_match_clean = (
    df_bronze_match_1.unionByName(
        df_bronze_match_2, allowMissingColumns=True
    )
    .dropDuplicates(["MatchId"])
)

#ensure schema overwrite is true 
df_match_clean.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("rugby_data_dev.rugby_silver.match_results")
